### Import Data

In [59]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print(os.getcwd())
df = pd.read_csv("../dataset/fab.csv")


d:\CODE\MyNote\SemiconductorData\code


### Data Information

In [ ]:
df.info()

# dtypes: float64(590) -> Sensor, int64(1) -> Pass_Fail, str(1) -> SensorTime
df.dtypes 

df.describe()

df.head()
df.tail()

### Set Target / Feature

In [60]:
target_col = "Pass_Fail"

feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

### Data Visualization

- 정상/불량 비율 확인

In [ ]:
print(df['Pass_Fail'].value_counts())

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

plt.figure(figsize=(5,5))
sns.countplot(x='Pass_Fail', data=df)
plt.title('정상 / 불량')
plt.show()

- 센서별 Pass / Fail 데이터 분포

In [ ]:
palette = {
    -1: "#4C72B0",   # Pass
     1: "#DD8452"    # Fail
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, sensor in enumerate(feature_cols[0:4]):
    sns.histplot(data=df,
                x=sensor,
                hue="Pass_Fail",
                bins=50,
                kde=True,
                palette=palette,
                alpha=0.45,
                edgecolor="black",
                linewidth=0.4,
                multiple="layer",
                ax=axes[i])
    sns.rugplot(data=df,
                x=sensor,
                hue="Pass_Fail",
                palette=palette,
                height=0.08,
                alpha=0.7,
                linewidth=1.2,
                ax=axes[i],
                legend=False)

    axes[i].set_title(f"{sensor} 데이터 분포")
    axes[i].set_xlabel("실제 측정값")
    axes[i].set_ylabel("데이터 수")

### Data Preprocessing

- 결측치 처리 

In [ ]:
# 결측값이 높은 feature 제거
missing_ratio = df[feature_cols].isnull().mean().sort_values(ascending=False) * 100
missing_ratio
high_missing_features = missing_ratio[missing_ratio > 70 ].index.tolist()
df = df.drop(columns=high_missing_features)

feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

# 나머지 결측치 데이터 중앙값으로 처리
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())
print(df.isna().sum().sum())
print(df[feature_cols].isna().mean().sort_values(ascending=False).head())

- 중복 데이터 처리

In [ ]:
duplicate_count = df.duplicated(subset=feature_cols + [target_col]).sum()
duplicate_count
# 중복 데이터 확인 결과, 전체 행 기준 중복 데이터는 존재하지 않았다.

- 이상치 처리

In [ ]:
outliers_result = {}
for col in feature_cols:
    Q1 = np.nanquantile(df[col], 0.25)
    Q3 = np.nanquantile(df[col], 0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_result[col] = len(outliers)
outliers_result = pd.Series(outliers_result).sort_values(ascending=False)
top_features = outliers_result.head(10).index
for col in top_features:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x='Pass_Fail', y=col)
    plt.title(f'{col} by Pass_Fail')
    plt.show()